# Final controller optimization and comprehensive evaluation

This notebook performs the final, evidence-driven evaluation of the existing
PI and LSTM-MPC controllers. It adds no models or diagnostic algorithms. The
study profiles SLSQP, tests warm starts, prediction/control horizons and move
blocking, selects the smallest stable configuration, then evaluates all four
controllers over eleven scenarios and five paired random seeds.

In [1]:
import cProfile
import json
import pstats
import sys
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from motor_model import DCMotorParams, dc_motor_dynamics
from mpc import (
    LSTMMPC,
    MPCConfig,
    PIConfig,
    PIController,
    RollingPredictionQuality,
    adaptation_region,
)
from reliability import SensorReliabilityMonitor, load_lstm_model

FIXED_FINAL_CONFIG_PATH = project_root / "results" / "configs" / "final_mpc_config.json"
fixed_final_config = json.loads(FIXED_FINAL_CONFIG_PATH.read_text(encoding="utf-8"))
SEED = 2026
FINAL_SEEDS = [int(seed) for seed in fixed_final_config["seed_repetitions"]]
assert FINAL_SEEDS == list(range(12026, 12031))
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(min(4, torch.get_num_threads()))

data = np.load(project_root / "data" / "processed" / "dc_motor_lstm_dataset.npz")
model, model_config = load_lstm_model(
    project_root / "results" / "lstm_model_weights.pt",
    project_root / "results" / "configs" / "lstm_model_config.json",
)
reliability_config = json.loads(
    (project_root / "results" / "configs" / "reliability_final_config.json").read_text()
)
lstm_metrics = json.loads(
    (project_root / "results" / "metrics" / "lstm_test_metrics.json").read_text()
)
normalization = model_config["normalization"]
window_length = int(model_config["window_length"])
DT = float(data["timestep"])
CONTROL_STRIDE = 5
CONTROL_DT = CONTROL_STRIDE * DT
DURATION = 6.0
TIME = np.arange(0.0, DURATION + DT / 2, DT)
FULL_SCALE = float(np.max(np.abs(data["y_true"])))
SPEED_NOISE_STD = float(data["speed_noise_std"])

PI_CONFIG = PIConfig(0.35, 0.8, (0.0, 12.0), 2.0)
CONTROLLERS = {
    "A_PI": "PI",
    "B_plain_MPC": "Plain LSTM-MPC",
    "C_sensor_MPC": "LSTM-MPC + sensor reliability",
    "D_adaptive": "Adaptive reliability-aware LSTM-MPC",
}
SCENARIOS = {
    "nominal_tracking": {"title": "Nominal tracking", "reference": "constant"},
    "step_reference": {"title": "Step reference", "reference": "step"},
    "changing_reference": {"title": "Changing reference", "reference": "changing"},
    "load_disturbance": {"title": "Load disturbance", "reference": "constant", "event_start": 3.0},
    "sensor_noise": {"title": "Sensor noise", "reference": "constant", "event_start": 2.0, "event_end": 4.0},
    "bias_5": {"title": "+5% sensor bias", "reference": "constant", "event_start": 2.0, "event_end": 4.0},
    "bias_15": {"title": "+15% sensor bias", "reference": "constant", "event_start": 2.0, "event_end": 4.0},
    "sensor_dropout": {"title": "Sensor dropout", "reference": "constant", "event_start": 2.0, "event_end": 4.0},
    "sensor_drift": {"title": "Sensor drift", "reference": "constant", "event_start": 2.0},
    "parameter_variation": {"title": "Parameter variation", "reference": "constant", "event_start": 3.0},
    "combined_fault_load": {"title": "Sensor fault + load disturbance", "reference": "constant", "event_start": 3.0},
}
SCENARIO_IDS = {name: index for index, name in enumerate(SCENARIOS)}

nominal_params = DCMotorParams()
shifted_params = replace(
    nominal_params,
    resistance=1.20 * nominal_params.resistance,
    inductance=0.85 * nominal_params.inductance,
    back_emf_constant=1.15 * nominal_params.back_emf_constant,
    torque_constant=0.85 * nominal_params.torque_constant,
    inertia=1.20 * nominal_params.inertia,
    viscous_friction=1.20 * nominal_params.viscous_friction,
    coulomb_friction=1.20 * nominal_params.coulomb_friction,
)


def reference_values(name, times=TIME):
    times = np.asarray(times)
    kind = SCENARIOS[name]["reference"]
    if kind == "step":
        return np.where(times < 1.5, 20.0, 40.0)
    if kind == "changing":
        return np.where(times < 2.0, 20.0, np.where(times < 4.0, 42.0, 30.0))
    return np.full(times.shape, 35.0)


def scenario_noise(name, seed):
    rng = np.random.default_rng(seed + 1009 * SCENARIO_IDS[name])
    return rng.normal(0.0, SPEED_NOISE_STD, len(TIME)), rng.normal(0.0, 2.0, len(TIME))


def load_torque(name, instant):
    disturbed = name in {"load_disturbance", "combined_fault_load"} and instant >= 3.0
    return 0.15 if disturbed else 0.03


def plant_params(name, instant):
    return shifted_params if name == "parameter_variation" and instant >= 3.0 else nominal_params


def measured_speed(name, index, true_speed, base_noise, added_noise):
    instant = TIME[index]
    measured = true_speed + base_noise[index]
    if name == "sensor_noise" and 2.0 <= instant < 4.0:
        measured += added_noise[index]
    elif name == "bias_5" and 2.0 <= instant < 4.0:
        measured += 0.05 * FULL_SCALE
    elif name == "bias_15" and 2.0 <= instant < 4.0:
        measured += 0.15 * FULL_SCALE
    elif name == "sensor_dropout" and 2.0 <= instant < 4.0:
        measured = 0.0
    elif name == "sensor_drift" and instant >= 2.0:
        measured += 0.15 * FULL_SCALE * min(1.0, (instant - 2.0) / 4.0)
    elif name == "combined_fault_load" and instant >= 3.0:
        measured += 0.05 * FULL_SCALE
    return float(measured)


def rk4_step(state, voltage, name, instant):
    params = plant_params(name, instant)
    load = load_torque(name, instant)
    derivative = lambda value: dc_motor_dynamics(instant, value, voltage, load, params)
    k1 = derivative(state)
    k2 = derivative(state + DT * k1 / 2)
    k3 = derivative(state + DT * k2 / 2)
    k4 = derivative(state + DT * k3)
    return state + DT * (k1 + 2 * k2 + 2 * k3 + k4) / 6


def blocks_of_size(horizon, width):
    return (width,) * (horizon // width) + ((horizon % width,) if horizon % width else ())


def simulate(
    controller_name,
    scenario_name,
    seed,
    mpc_config,
    *,
    adaptive_regions=None,
    model_scale=1.0,
    medium_threshold=np.inf,
    high_threshold=np.inf,
    warm_start=None,
    force_fallback_updates=(),
):
    reference = reference_values(scenario_name)
    base_noise, added_noise = scenario_noise(scenario_name, seed)
    state = np.zeros(2)
    previous_voltage = 0.0
    history = np.zeros((window_length, 2), dtype=np.float32)
    pi = PIController(PI_CONFIG)
    mpc = LSTMMPC(model, normalization, window_length, mpc_config)
    sensor = SensorReliabilityMonitor(
        reliability_config["sensor"]["instant_threshold"],
        reliability_config["sensor"]["center"],
        reliability_config["sensor"]["allowance"],
        reliability_config["sensor"]["threshold"],
        reliability_config["sensor"]["enter_count"],
        reliability_config["sensor"]["exit_count"],
    )
    quality = RollingPredictionQuality(model_scale, rolling_window=20, horizon=CONTROL_STRIDE)
    fields = [
        "true", "measured", "virtual", "feedback", "voltage", "error",
        "sensor_state", "substituted", "sensor_score", "positive_cusum",
        "negative_cusum", "healthy_run", "model_quality", "used_fallback",
        "horizon", "control_horizon", "mode", "optimizer_success", "compute_ms",
        "iterations", "objective_evaluations", "active_max_step",
    ]
    arrays = {name: np.zeros(len(TIME), dtype=float) for name in fields}
    for name in ["optimizer_success", "compute_ms", "iterations", "objective_evaluations"]:
        arrays[name][:] = np.nan
    mode_code = {"LOW": 0, "MEDIUM": 1, "HIGH": 2}
    safety_events = 0
    nonfinite_events = 0

    for index, instant in enumerate(TIME):
        true_speed = float(state[1])
        raw_y_measured = measured_speed(scenario_name, index, true_speed, base_noise, added_noise)
        if not np.isfinite(raw_y_measured):
            nonfinite_events += 1
        try:
            y_hat = float(mpc.predict(history, np.array([previous_voltage], dtype=np.float32))[0])
        except (ValueError, FloatingPointError):
            y_hat = raw_y_measured if np.isfinite(raw_y_measured) else true_speed
            safety_events += 1
            nonfinite_events += 1

        reliability_enabled = controller_name in {"C_sensor_MPC", "D_adaptive"} and index >= window_length
        # Health always uses the raw physical measurement; only feedback/history may be substituted.
        reliability = sensor.update(raw_y_measured - y_hat) if reliability_enabled else {
            "trusted": np.isfinite(raw_y_measured), "sensor_suspect": False,
            "substitute": not np.isfinite(raw_y_measured), "score": 0.0,
            "positive_cusum": 0.0, "negative_cusum": 0.0, "healthy_run": 0,
        }
        y_feedback = y_hat if reliability["substitute"] else raw_y_measured
        if controller_name == "D_adaptive" and index >= window_length:
            quality_score = quality.update(raw_y_measured, reliability["trusted"])
        else:
            quality_score = quality.score
        if instant < 1.0:
            quality_score = 1.0
        region = (
            adaptation_region(quality_score, medium_threshold, high_threshold)
            if controller_name == "D_adaptive" else "LOW"
        )
        settings = adaptive_regions[region] if controller_name == "D_adaptive" else {
            "horizon": mpc_config.horizon,
            "control_horizon": mpc_config.control_horizon,
            "move_blocks": mpc_config.move_blocks,
            "move_weight": mpc_config.move_weight,
            "max_voltage_step": mpc_config.max_voltage_step,
        }

        history = np.vstack((history[1:], [previous_voltage, y_feedback])).astype(np.float32)
        voltage = previous_voltage
        if index % CONTROL_STRIDE == 0:
            fallback = pi.compute_control(reference[index], y_feedback, CONTROL_DT)
            if controller_name == "A_PI":
                voltage = fallback
            else:
                horizon = settings["horizon"]
                future_time = instant + DT * np.arange(1, horizon + 1)
                forced_fallback = index // CONTROL_STRIDE in force_fallback_updates
                output = mpc.compute_control(
                    np.full_like(history, np.nan) if forced_fallback else history,
                    reference_values(scenario_name, future_time),
                    previous_voltage,
                    horizon=horizon,
                    control_horizon=settings.get("control_horizon"),
                    move_blocks=settings.get("move_blocks"),
                    warm_start=warm_start,
                    move_weight=settings["move_weight"],
                    max_voltage_step=settings["max_voltage_step"],
                    fallback_voltage=fallback,
                )
                voltage = output["voltage"]
                arrays["optimizer_success"][index] = float(output["success"])
                arrays["compute_ms"][index] = output["compute_ms"]
                arrays["iterations"][index] = output["iterations"]
                arrays["objective_evaluations"][index] = output["objective_evaluations"]
                arrays["used_fallback"][index] = output["used_fallback"]
                if not np.isfinite(output["compute_ms"]):
                    nonfinite_events += 1
                if controller_name == "D_adaptive":
                    quality.add_forecast(output["prediction"][:CONTROL_STRIDE])
            if not np.isfinite(voltage):
                nonfinite_events += 1
            if not np.isfinite(voltage) or abs(true_speed) > 120.0:
                voltage = fallback if np.isfinite(fallback) else previous_voltage
                safety_events += 1
            voltage = float(np.clip(
                voltage,
                max(0.0, previous_voltage - settings["max_voltage_step"]),
                min(12.0, previous_voltage + settings["max_voltage_step"]),
            ))
        history[-1, 0] = voltage

        arrays["true"][index] = true_speed
        arrays["measured"][index] = raw_y_measured
        arrays["virtual"][index] = y_hat
        arrays["feedback"][index] = y_feedback
        arrays["voltage"][index] = voltage
        arrays["error"][index] = reference[index] - true_speed
        arrays["sensor_state"][index] = reliability["sensor_suspect"]
        arrays["substituted"][index] = reliability["substitute"]
        arrays["sensor_score"][index] = reliability["score"]
        arrays["positive_cusum"][index] = reliability["positive_cusum"]
        arrays["negative_cusum"][index] = reliability["negative_cusum"]
        arrays["healthy_run"][index] = reliability["healthy_run"]
        arrays["model_quality"][index] = quality_score
        arrays["horizon"][index] = settings["horizon"] if controller_name != "A_PI" else 0
        arrays["control_horizon"][index] = (
            len(settings["move_blocks"]) if settings.get("move_blocks")
            else settings.get("control_horizon") or settings["horizon"]
        ) if controller_name != "A_PI" else 0
        arrays["mode"][index] = mode_code[region] if controller_name == "D_adaptive" else 0
        arrays["active_max_step"][index] = settings["max_voltage_step"]

        previous_voltage = voltage
        if index < len(TIME) - 1:
            state = rk4_step(state, voltage, scenario_name, instant)
            if not np.isfinite(state).all():
                raise FloatingPointError(f"non-finite plant state in {controller_name}/{scenario_name}")

    arrays["time"] = TIME.copy()
    arrays["reference"] = reference
    arrays["safety_events"] = safety_events
    arrays["nonfinite_events"] = nonfinite_events
    return arrays

In [2]:
def sustained_time(trace, start):
    start_index = int(np.searchsorted(TIME, start))
    band = np.maximum(1.0, 0.05 * np.abs(trace["reference"]))
    within = np.abs(trace["error"]) <= band
    dwell = int(round(0.25 / DT))
    for index in range(start_index, len(TIME) - dwell + 1):
        if np.all(within[index:index + dwell]):
            return float(TIME[index] - start)
    return np.nan


def settling_time(trace):
    changes = np.flatnonzero(np.abs(np.diff(trace["reference"])) > 1e-9) + 1
    start = float(TIME[changes[-1]]) if len(changes) else 0.0
    return sustained_time(trace, start)


def metrics(trace, controller, scenario, seed):
    attempts = np.isfinite(trace["optimizer_success"])
    solve_ms = trace["compute_ms"][attempts]
    control_indices = np.arange(0, len(TIME), CONTROL_STRIDE)
    control_voltage = trace["voltage"][control_indices]
    control_moves = np.diff(np.r_[0.0, control_voltage])
    allowed_moves = trace["active_max_step"][control_indices]
    voltage_violations = int(np.sum((trace["voltage"] < -1e-8) | (trace["voltage"] > 12.0 + 1e-8)))
    rate_violations = int(np.sum(np.abs(control_moves) > allowed_moves + 1e-8))
    scenario_config = SCENARIOS[scenario]
    event_start = scenario_config.get("event_start")
    event_end = scenario_config.get("event_end")
    if event_start is None:
        event_mask = np.ones(len(TIME), dtype=bool)
        recovery = settling_time(trace)
    else:
        event_mask = TIME >= event_start
        if event_end is not None:
            event_mask &= TIME < event_end
        recovery = sustained_time(trace, event_end if event_end is not None else event_start)
    active = trace["sensor_state"].astype(bool)
    recovery_indices = np.flatnonzero(active[:-1] & ~active[1:]) + 1
    sensor_recovered = np.flatnonzero((TIME >= event_end) & ~active) if event_end is not None and np.any(active & event_mask) else []
    error = trace["error"]
    changes = np.flatnonzero(np.abs(np.diff(trace["reference"])) > 1e-9) + 1
    final_start = int(changes[-1]) if len(changes) else 0
    final_reference = trace["reference"][-1]
    overshoot = 100 * max(0.0, float(np.max(trace["true"][final_start:] - final_reference))) / max(abs(final_reference), 1e-9)
    return {
        "controller": controller,
        "scenario": scenario,
        "seed": seed,
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(np.abs(error))),
        "IAE": float(np.trapezoid(np.abs(error), TIME)),
        "ISE": float(np.trapezoid(error ** 2, TIME)),
        "max_abs_error": float(np.max(np.abs(error))),
        "overshoot_percent": overshoot,
        "settling_time_s": settling_time(trace),
        "recovery_time_s": recovery,
        "fault_interval_RMSE": float(np.sqrt(np.mean(error[event_mask] ** 2))),
        "fault_interval_IAE": float(np.trapezoid(np.abs(error[event_mask]), TIME[event_mask])),
        "fault_interval_ISE": float(np.trapezoid(error[event_mask] ** 2, TIME[event_mask])),
        "fault_interval_peak_error": float(np.max(np.abs(error[event_mask]))),
        "control_effort_sum_u2": float(np.sum(control_voltage ** 2)),
        "control_variation_sum_du2": float(np.sum(control_moves ** 2)),
        "constraint_violations": voltage_violations + rate_violations,
        "voltage_violations": voltage_violations,
        "rate_violations": rate_violations,
        "nonfinite_events": int(trace["nonfinite_events"]),
        "optimizer_failures": int(np.sum(trace["optimizer_success"][attempts] < 0.5)),
        "average_solve_ms": float(np.mean(solve_ms)) if len(solve_ms) else 0.0,
        "p95_solve_ms": float(np.percentile(solve_ms, 95)) if len(solve_ms) else 0.0,
        "max_solve_ms": float(np.max(solve_ms)) if len(solve_ms) else 0.0,
        "mean_iterations": float(np.nanmean(trace["iterations"])) if np.any(attempts) else 0.0,
        "mean_objective_evaluations": float(np.nanmean(trace["objective_evaluations"])) if np.any(attempts) else 0.0,
        "tail_oscillation": float(np.ptp(trace["true"][TIME >= DURATION - 1.0])),
        "sensor_substitution_rate": float(np.mean(trace["substituted"])),
        "substitution_fraction": float(np.mean(trace["substituted"])),
        "recovery_events": int(len(recovery_indices)),
        "false_recovery_events": int(np.sum(event_mask[recovery_indices])) if event_start is not None else 0,
        "sensor_recovery_latency_s": float(TIME[sensor_recovered[0]] - event_end) if len(sensor_recovered) else np.nan,
        "medium_mode_fraction": float(np.mean(trace["mode"] == 1)),
        "high_mode_fraction": float(np.mean(trace["mode"] == 2)),
        "safety_events": int(trace["safety_events"]),
    }


METRIC_COLUMNS = [
    "RMSE", "MAE", "IAE", "ISE", "max_abs_error", "overshoot_percent",
    "settling_time_s", "recovery_time_s", "fault_interval_RMSE",
    "fault_interval_IAE", "fault_interval_ISE", "fault_interval_peak_error",
    "control_effort_sum_u2", "control_variation_sum_du2", "constraint_violations",
    "voltage_violations", "rate_violations", "nonfinite_events",
    "optimizer_failures", "average_solve_ms", "p95_solve_ms", "max_solve_ms",
    "tail_oscillation", "sensor_substitution_rate", "substitution_fraction",
    "recovery_events", "false_recovery_events", "sensor_recovery_latency_s", "medium_mode_fraction",
    "high_mode_fraction", "safety_events",
]


def aggregate_runs(frame):
    grouped = frame.groupby(["controller", "scenario"], sort=True)
    means = grouped[METRIC_COLUMNS].mean()
    stds = grouped[METRIC_COLUMNS].std(ddof=1).add_suffix("_std")
    result = means.join(stds).reset_index()
    result.insert(2, "repetitions", grouped.size().to_numpy())
    return result

## Runtime profile and horizon/control-horizon study

The profile is taken on a cold, full-sequence `H=20` solve. The systematic
study then runs every valid combination of `H in {5, 10, 15, 20}` and
valid 50 ms-aligned control horizons on nominal tracking, a reference step and a load step.

In [3]:
profile_config = MPCConfig(horizon=20, control_horizon=4, move_weight=0.5, max_iterations=25, tolerance=5e-2, control_interval_steps=CONTROL_STRIDE)
profile_controller = LSTMMPC(model, normalization, window_length, profile_config)
profile_history = np.column_stack((
    np.full(window_length, 6.0, dtype=np.float32),
    np.linspace(25.0, 35.0, window_length, dtype=np.float32),
))
profiler = cProfile.Profile()
profile_output = profiler.runcall(profile_controller.compute_control, profile_history, 35.0, 6.0)
profile_stats = pstats.Stats(profiler)
profile_rows = []
for (filename, line, function), (_, calls, own_time, cumulative_time, _) in profile_stats.stats.items():
    if function in {"compute_control", "_minimize_slsqp", "objective", "_rollout", "_engine_run_backward"}:
        profile_rows.append({
            "study": "profile",
            "component": function,
            "calls": calls,
            "own_ms": 1000 * own_time,
            "cumulative_ms": 1000 * cumulative_time,
        })
profile_df = pd.DataFrame(profile_rows).sort_values("cumulative_ms", ascending=False)
display(profile_df.round(2))

horizon_rows = []
horizon_scenarios = ["nominal_tracking", "step_reference", "load_disturbance"]
combinations = [
    (horizon, control_horizon)
    for horizon in [5, 10, 15, 20]
    for control_horizon in range(1, (horizon - 1) // CONTROL_STRIDE + 2)
]
for run_number, (horizon, control_horizon, scenario) in enumerate(
    [(h, nc, s) for h, nc in combinations for s in horizon_scenarios], 1
):
    print(f"[horizon {run_number:02d}/{len(combinations) * len(horizon_scenarios)}] H={horizon}, Nc={control_horizon}, {scenario}")
    config = MPCConfig(
        horizon=horizon,
        control_horizon=control_horizon,
        move_weight=0.5,
        max_iterations=25,
        tolerance=5e-2,
        control_interval_steps=CONTROL_STRIDE,
    )
    trace = simulate("B_plain_MPC", scenario, SEED, config)
    row = metrics(trace, "B_plain_MPC", scenario, SEED)
    row.update(study="horizon_control", horizon=horizon, control_horizon=control_horizon, warm_start=True)
    horizon_rows.append(row)

horizon_df = pd.DataFrame(horizon_rows)
horizon_summary = horizon_df.groupby(["horizon", "control_horizon"], as_index=False).agg(
    RMSE=("RMSE", "mean"),
    ISE=("ISE", "mean"),
    tail_oscillation=("tail_oscillation", "mean"),
    control_effort_sum_u2=("control_effort_sum_u2", "mean"),
    control_variation_sum_du2=("control_variation_sum_du2", "mean"),
    average_solve_ms=("average_solve_ms", "mean"),
    p95_solve_ms=("p95_solve_ms", "mean"),
    max_solve_ms=("max_solve_ms", "max"),
    optimizer_failures=("optimizer_failures", "sum"),
    constraint_violations=("constraint_violations", "sum"),
)
valid = horizon_summary[
    (horizon_summary.optimizer_failures == 0)
    & (horizon_summary.constraint_violations == 0)
].copy()
for column in ["RMSE", "tail_oscillation", "average_solve_ms"]:
    valid[f"normalized_{column}"] = valid[column] / max(valid[column].min(), 1e-9)
valid["selection_score"] = (
    valid["normalized_RMSE"]
    + 0.25 * valid["normalized_tail_oscillation"]
    + 0.25 * valid["normalized_average_solve_ms"]
)
selected_horizon_row = valid.loc[valid.selection_score.idxmin()]
SELECTED_H = int(selected_horizon_row.horizon)
SELECTED_NC = int(selected_horizon_row.control_horizon)
display(horizon_summary.sort_values(["selection_score"] if "selection_score" in horizon_summary else ["horizon", "control_horizon"]).round(3))
print(f"Initial compromise selected: H={SELECTED_H}, Nc={SELECTED_NC}")

,study,component,calls,own_ms,cumulative_ms
4,profile,compute_control,1,0.29,290.50
2,profile,_minimize_slsqp,1,0.65,265.92
3,profile,objective,10,7.20,261.22
1,profile,_rollout,5,14.34,141.83
0,profile,_engine_run_backward,5,0.14,104.86


[horizon 01/30] H=5, Nc=1, nominal_tracking


[horizon 02/30] H=5, Nc=1, step_reference


[horizon 03/30] H=5, Nc=1, load_disturbance


[horizon 04/30] H=10, Nc=1, nominal_tracking


...

[horizon 28/30] H=20, Nc=4, nominal_tracking


[horizon 29/30] H=20, Nc=4, step_reference


[horizon 30/30] H=20, Nc=4, load_disturbance


,horizon,control_horizon,RMSE,ISE,tail_oscillation,control_effort_sum_u2,control_variation_sum_du2,average_solve_ms,p95_solve_ms,max_solve_ms,optimizer_failures,constraint_violations
0,5,1,29.081,5114.605,0.993,16924.000,24.000,8.878,14.248,25.971,0,0
1,10,1,10.272,644.613,9.827,8878.128,172.921,22.724,37.437,46.911,0,0
2,10,2,10.305,648.653,10.198,8943.118,170.953,36.858,75.061,102.707,0,0
3,15,1,9.772,584.022,3.577,8463.838,252.252,39.383,62.215,86.348,0,0
4,15,2,9.776,584.420,3.689,8483.696,253.241,57.774,105.523,133.482,0,0
5,15,3,9.786,585.605,3.877,8559.771,249.663,70.745,128.536,199.180,0,0
6,20,1,9.565,563.917,0.409,7320.235,178.028,70.379,113.585,142.971,0,0
7,20,2,9.566,564.070,0.468,7338.255,170.705,87.138,144.462,302.811,0,0
8,20,3,9.566,564.094,0.441,7336.956,167.609,105.872,190.314,241.437,0,0
9,20,4,9.566,564.110,0.448,7340.615,167.334,114.486,224.424,304.059,0,0


Initial compromise selected: H=20, Nc=1


## Warm start, move blocking and iteration cap

Warm starts are compared with constant initialization. Move blocking includes
the selected hold-last control horizon, a full sequence, approximately five
uniform moves, and blocks aligned with the 50 ms control interval. The
smallest iteration cap within 1% of the 25-iteration reference is retained.

In [4]:
study_scenarios = ["nominal_tracking", "step_reference", "load_disturbance"]
warm_rows = []
for enabled in [False, True]:
    config = MPCConfig(
        horizon=SELECTED_H,
        control_horizon=SELECTED_NC,
        move_weight=0.5,
        max_iterations=25,
        tolerance=5e-2,
        warm_start=enabled,
        control_interval_steps=CONTROL_STRIDE,
    )
    for scenario in study_scenarios:
        trace = simulate("B_plain_MPC", scenario, SEED, config, warm_start=enabled)
        row = metrics(trace, "B_plain_MPC", scenario, SEED)
        row.update(study="warm_start", horizon=SELECTED_H, control_horizon=SELECTED_NC, warm_start=enabled)
        warm_rows.append(row)
warm_df = pd.DataFrame(warm_rows)
warm_summary = warm_df.groupby("warm_start", as_index=False).agg(
    RMSE=("RMSE", "mean"),
    tail_oscillation=("tail_oscillation", "mean"),
    average_solve_ms=("average_solve_ms", "mean"),
    p95_solve_ms=("p95_solve_ms", "mean"),
    optimizer_failures=("optimizer_failures", "sum"),
)
display(warm_summary.round(3))

strategy_specs = {
    "selected_multirate_control_horizon": {"control_horizon": SELECTED_NC, "move_blocks": None},
    "control_interval_blocks": {"control_horizon": None, "move_blocks": blocks_of_size(SELECTED_H, CONTROL_STRIDE)},
}
move_rows = []
for strategy, spec in strategy_specs.items():
    config = MPCConfig(
        horizon=SELECTED_H,
        control_horizon=spec["control_horizon"],
        move_blocks=spec["move_blocks"],
        move_weight=0.5,
        max_iterations=25,
        tolerance=5e-2,
        control_interval_steps=CONTROL_STRIDE,
    )
    for scenario in study_scenarios:
        trace = simulate("B_plain_MPC", scenario, SEED, config)
        row = metrics(trace, "B_plain_MPC", scenario, SEED)
        row.update(
            study="move_blocking",
            strategy=strategy,
            horizon=SELECTED_H,
            control_horizon=len(spec["move_blocks"]) if spec["move_blocks"] else spec["control_horizon"],
            move_blocks=str(spec["move_blocks"]),
            warm_start=True,
        )
        move_rows.append(row)
move_df = pd.DataFrame(move_rows)
move_summary = move_df.groupby("strategy", as_index=False).agg(
    RMSE=("RMSE", "mean"),
    tail_oscillation=("tail_oscillation", "mean"),
    average_solve_ms=("average_solve_ms", "mean"),
    p95_solve_ms=("p95_solve_ms", "mean"),
    optimizer_failures=("optimizer_failures", "sum"),
    constraint_violations=("constraint_violations", "sum"),
)
baseline_strategy = move_summary[move_summary.strategy == "selected_multirate_control_horizon"].iloc[0]
eligible_strategies = move_summary[
    (move_summary.optimizer_failures == 0)
    & (move_summary.constraint_violations == 0)
    & (move_summary.RMSE <= 1.03 * baseline_strategy.RMSE)
    & (move_summary.tail_oscillation <= max(1.10 * baseline_strategy.tail_oscillation, baseline_strategy.tail_oscillation + 0.05))
]
selected_strategy = eligible_strategies.sort_values("average_solve_ms").iloc[0].strategy
selected_spec = strategy_specs[selected_strategy]
display(move_summary.round(3))
print("Selected blocking strategy:", selected_strategy, selected_spec)

iteration_rows = []
for max_iterations in [8, 12, 25]:
    config = MPCConfig(
        horizon=SELECTED_H,
        control_horizon=selected_spec["control_horizon"],
        move_blocks=selected_spec["move_blocks"],
        move_weight=0.5,
        max_iterations=max_iterations,
        tolerance=5e-2,
        control_interval_steps=CONTROL_STRIDE,
    )
    for scenario in study_scenarios:
        trace = simulate("B_plain_MPC", scenario, SEED, config)
        row = metrics(trace, "B_plain_MPC", scenario, SEED)
        row.update(study="max_iterations", max_iterations=max_iterations, horizon=SELECTED_H, strategy=selected_strategy)
        iteration_rows.append(row)
iteration_df = pd.DataFrame(iteration_rows)
iteration_summary = iteration_df.groupby("max_iterations", as_index=False).agg(
    RMSE=("RMSE", "mean"),
    tail_oscillation=("tail_oscillation", "mean"),
    average_solve_ms=("average_solve_ms", "mean"),
    p95_solve_ms=("p95_solve_ms", "mean"),
    optimizer_failures=("optimizer_failures", "sum"),
)
reference_iterations = iteration_summary[iteration_summary.max_iterations == 25].iloc[0]
eligible_iterations = iteration_summary[
    (iteration_summary.optimizer_failures == 0)
    & (iteration_summary.RMSE <= 1.01 * reference_iterations.RMSE)
    & (iteration_summary.tail_oscillation <= max(1.05 * reference_iterations.tail_oscillation, reference_iterations.tail_oscillation + 0.02))
]
SELECTED_MAX_ITERATIONS = int(eligible_iterations.max_iterations.min())
display(iteration_summary.round(3))
print("Selected max iterations:", SELECTED_MAX_ITERATIONS)

,warm_start,RMSE,tail_oscillation,average_solve_ms,p95_solve_ms,optimizer_failures
0,False,9.565,0.409,74.134,117.750,0
1,True,9.565,0.409,71.956,117.753,0


,strategy,RMSE,tail_oscillation,average_solve_ms,p95_solve_ms,optimizer_failures,constraint_violations
0,control_interval_blocks,9.566,0.448,98.610,186.503,0,0
1,selected_multirate_control_horizon,9.565,0.409,54.449,88.682,0,0


Selected blocking strategy: selected_multirate_control_horizon {'control_horizon': 1, 'move_blocks': None}


,max_iterations,RMSE,tail_oscillation,average_solve_ms,p95_solve_ms,optimizer_failures
0,8,9.565,0.409,55.043,91.955,0
1,12,9.565,0.409,58.563,95.724,0
2,25,9.565,0.409,67.256,117.488,0


Selected max iterations: 8


## Gradual model-quality adaptation and five-seed final matrix

The previous medium/high settings (`R = 1/2`, slew `= 1/0.5 V`) were overly
conservative. The final schedule keeps the selected low-mismatch controller,
increases the move penalty gradually, and only shortens the high-mismatch
horizon by five model steps. PI is reserved for optimizer or safety failure.

In [5]:
mpc_values = dict(fixed_final_config["mpc"])
mpc_values["move_blocks"] = tuple(mpc_values["move_blocks"])
FINAL_CONFIG = MPCConfig(**mpc_values)
assert FINAL_CONFIG.horizon == 20
assert FINAL_CONFIG.move_blocks == (5, 15)
assert FINAL_CONFIG.control_interval_steps == CONTROL_STRIDE == 5
assert FINAL_CONFIG.move_blocks[0] == FINAL_CONFIG.control_interval_steps

quality_config = fixed_final_config["model_quality"]
MODEL_SCALE = float(quality_config["scale_median_rms"])
MEDIUM_THRESHOLD = float(quality_config["medium_threshold_p75"])
HIGH_THRESHOLD = float(quality_config["high_threshold_p95"])
ADAPTIVE_REGIONS = {
    name: {**values, "move_blocks": tuple(values["move_blocks"]) if values["move_blocks"] else None}
    for name, values in quality_config["regions"].items()
}
assert all(region["move_blocks"][0] == CONTROL_STRIDE for region in ADAPTIVE_REGIONS.values())

final_rows = []
final_runtime_rows = []
virtual_feedback_rows = []
representative_traces = {}
sensor_fault_scenarios = {"sensor_noise", "bias_5", "bias_15", "sensor_dropout", "sensor_drift", "combined_fault_load"}
run_matrix = [
    (controller, scenario, seed)
    for seed in FINAL_SEEDS
    for scenario in SCENARIOS
    for controller in CONTROLLERS
]
for run_number, (controller, scenario, seed) in enumerate(run_matrix, 1):
    if run_number == 1 or run_number % 10 == 0:
        print(f"[final {run_number:03d}/{len(run_matrix)}] seed={seed}, {controller}, {scenario}")
    trace = simulate(
        controller,
        scenario,
        seed,
        FINAL_CONFIG,
        adaptive_regions=ADAPTIVE_REGIONS,
        model_scale=MODEL_SCALE,
        medium_threshold=MEDIUM_THRESHOLD,
        high_threshold=HIGH_THRESHOLD,
    )
    final_rows.append(metrics(trace, controller, scenario, seed))
    if controller in {"C_sensor_MPC", "D_adaptive"} and scenario in sensor_fault_scenarios:
        mask = trace["substituted"].astype(bool)
        virtual_error = trace["virtual"][mask] - trace["true"][mask]
        measured_error = trace["measured"][mask] - trace["true"][mask]
        event_end = SCENARIOS[scenario].get("event_end")
        recovered = np.flatnonzero((TIME >= event_end) & ~mask) if event_end is not None and np.any(mask) else []
        # y_true is used here only for offline validation; it never enters monitor/control logic.
        virtual_feedback_rows.append({
            "controller": controller, "scenario": scenario, "seed": seed,
            "virtual_feedback_RMSE": float(np.sqrt(np.mean(virtual_error ** 2))) if len(virtual_error) else np.nan,
            "corrupted_sensor_RMSE": float(np.sqrt(np.mean(measured_error ** 2))) if len(measured_error) else np.nan,
            "virtual_feedback_MAE": float(np.mean(np.abs(virtual_error))) if len(virtual_error) else np.nan,
            "corrupted_sensor_MAE": float(np.mean(np.abs(measured_error))) if len(measured_error) else np.nan,
            "substitution_duration_s": float(np.sum(mask) * DT),
            "recovery_latency_s": float(TIME[recovered[0]] - event_end) if len(recovered) else np.nan,
        })
    for sample_index in np.flatnonzero(np.isfinite(trace["optimizer_success"])):
        final_runtime_rows.append({
            "run_number": run_number,
            "controller": controller,
            "scenario": scenario,
            "seed": seed,
            "sample_index": int(sample_index),
            "control_step": int(sample_index // CONTROL_STRIDE),
            "time_s": float(TIME[sample_index]),
            "compute_ms": float(trace["compute_ms"][sample_index]),
            "optimizer_success": bool(trace["optimizer_success"][sample_index]),
        })
    if seed == FINAL_SEEDS[0]:
        representative_traces[(controller, scenario)] = trace

final_runs_df = pd.DataFrame(final_rows)
final_comparison_df = aggregate_runs(final_runs_df)
final_runtime_samples_df = pd.DataFrame(final_runtime_rows)
virtual_feedback_quality_df = pd.DataFrame(virtual_feedback_rows)
forced_updates = range(int(3.0 / CONTROL_DT), int(3.0 / CONTROL_DT) + 10)
fallback_trace = simulate(
    "D_adaptive", "parameter_variation", FINAL_SEEDS[0], FINAL_CONFIG,
    adaptive_regions=ADAPTIVE_REGIONS, model_scale=MODEL_SCALE,
    medium_threshold=MEDIUM_THRESHOLD, high_threshold=HIGH_THRESHOLD,
    force_fallback_updates=forced_updates,
)
control_indices = np.arange(0, len(TIME), CONTROL_STRIDE)
forced_indices = control_indices[list(forced_updates)]
resume_index = control_indices[list(forced_updates)[-1] + 1]
control_moves = np.diff(np.r_[0.0, fallback_trace["voltage"][control_indices]])
fallback_validation = {
    "forced_consecutive_updates": int(np.sum(fallback_trace["used_fallback"][forced_indices])),
    "max_abs_speed_rad_s": float(np.max(np.abs(fallback_trace["true"]))),
    "voltage_violations": int(np.sum((fallback_trace["voltage"] < 0) | (fallback_trace["voltage"] > 12))),
    "slew_violations": int(np.sum(np.abs(control_moves) > fallback_trace["active_max_step"][control_indices] + 1e-8)),
    "mpc_resumed": bool(fallback_trace["optimizer_success"][resume_index] == 1),
    "resume_voltage_jump_v": float(abs(fallback_trace["voltage"][resume_index] - fallback_trace["voltage"][control_indices[list(forced_updates)[-1]]])),
    "resume_slew_limit_v": float(fallback_trace["active_max_step"][resume_index]),
}
assert fallback_validation["forced_consecutive_updates"] == 10
assert fallback_validation["max_abs_speed_rad_s"] <= 120.0
assert fallback_validation["voltage_violations"] == fallback_validation["slew_violations"] == 0
assert fallback_validation["mpc_resumed"] and fallback_validation["resume_voltage_jump_v"] <= fallback_validation["resume_slew_limit_v"] + 1e-8
display(final_comparison_df[[
    "controller", "scenario", "repetitions", "RMSE", "RMSE_std",
    "fault_interval_RMSE", "fault_interval_RMSE_std", "recovery_time_s",
    "control_variation_sum_du2", "constraint_violations", "optimizer_failures",
    "average_solve_ms", "p95_solve_ms", "tail_oscillation",
]].round(3))

[final 001/220] seed=12026, A_PI, nominal_tracking


[final 010/220] seed=12026, B_plain_MPC, changing_reference


[final 020/220] seed=12026, D_adaptive, sensor_noise


[final 030/220] seed=12026, B_plain_MPC, sensor_dropout


[final 040/220] seed=12026, D_adaptive, parameter_variation


...

[final 190/220] seed=12030, B_plain_MPC, load_disturbance


[final 200/220] seed=12030, D_adaptive, bias_5


[final 210/220] seed=12030, B_plain_MPC, sensor_drift


[final 220/220] seed=12030, D_adaptive, combined_fault_load


,controller,scenario,repetitions,RMSE,RMSE_std,fault_interval_RMSE,fault_interval_RMSE_std,recovery_time_s,control_variation_sum_du2,constraint_violations,optimizer_failures,average_solve_ms,p95_solve_ms,tail_oscillation
0,A_PI,bias_15,5,12.892,0.016,10.805,0.076,0.768,39.688,0.0,0.0,0.000,0.000,4.062
1,A_PI,bias_5,5,11.436,0.013,4.995,0.031,0.164,30.113,0.0,0.0,0.000,0.000,2.091
2,A_PI,changing_reference,5,9.706,0.035,9.706,0.035,NaN,43.955,0.0,0.0,0.000,0.000,5.604
3,A_PI,combined_fault_load,5,11.569,0.016,4.650,0.036,1.758,28.160,0.0,0.0,0.000,0.000,2.243
4,A_PI,load_disturbance,5,11.284,0.012,2.872,0.034,1.076,27.450,0.0,0.0,0.000,0.000,2.456
5,A_PI,nominal_tracking,5,11.134,0.010,11.134,0.010,2.332,25.953,0.0,0.0,0.000,0.000,0.662
6,A_PI,parameter_variation,5,11.201,0.010,2.090,0.039,1.208,26.538,0.0,0.0,0.000,0.000,0.382
7,A_PI,sensor_drift,5,11.888,0.017,5.363,0.055,0.318,26.021,0.0,0.0,0.000,0.000,2.951
8,A_PI,sensor_dropout,5,19.127,0.031,20.005,0.052,NaN,65.832,0.0,0.0,0.000,0.000,13.278
9,A_PI,sensor_noise,5,11.124,0.025,2.137,0.231,0.104,61.456,0.0,0.0,0.000,0.000,0.586


## Final artifacts, representative plots, and conclusions

### Submission interpretation

The corrective final evaluation loads the frozen configuration from `results/configs/final_mpc_config.json`; development timing studies above cannot select or alter it. The H=20, Nc=2 plan uses `(5, 15)` model-step blocks, so the first optimized command is held for the same 50 ms interval used by the plant simulation.

All headline values below are derived only from the fresh deterministic final holdout seeds (12026-12030). Historical percentages and runtime baselines are intentionally not reused; the regenerated artifacts report whatever the corrected experiment produces.

> Run this notebook sequentially after the corrected MPC core is installed. Saved notebook outputs remain historical until that corrective run completes; canonical status comes from the regenerated final artifacts.

In [6]:
metrics_dir = project_root / "results" / "metrics"
plots_dir = project_root / "results" / "plots"
configs_dir = project_root / "results" / "configs"
raw_dir = project_root / "results" / "raw"
metrics_dir.mkdir(parents=True, exist_ok=True)
plots_dir.mkdir(parents=True, exist_ok=True)
configs_dir.mkdir(parents=True, exist_ok=True)
raw_dir.mkdir(parents=True, exist_ok=True)

final_runs_df.to_csv(metrics_dir / "final_controller_runs.csv", index=False)
final_comparison_df.to_csv(metrics_dir / "final_controller_comparison.csv", index=False)
final_runtime_samples_df.to_csv(metrics_dir / "final_runtime_samples.csv", index=False)
virtual_feedback_quality_df.to_csv(metrics_dir / "virtual_feedback_quality.csv", index=False)
(metrics_dir / "pi_fallback_validation.json").write_text(json.dumps(fallback_validation, indent=2), encoding="utf-8")
trace_output = {
    f"{controller}__{scenario}__{field}": values
    for (controller, scenario), trace in representative_traces.items()
    for field, values in trace.items()
}
np.savez_compressed(raw_dir / "final_representative_traces.npz", **trace_output)

def validation_row(phase, scenario, trace):
    row = metrics(trace, "D_adaptive", scenario, FINAL_SEEDS[0])
    event_start = SCENARIOS[scenario]["event_start"]
    detections = np.flatnonzero((TIME >= event_start) & trace["sensor_state"].astype(bool))
    return {
        "phase": phase, "scenario": scenario,
        "detection_latency_s": float(TIME[detections[0]] - event_start) if len(detections) else np.nan,
        "tracking_RMSE": row["RMSE"], "fault_window_RMSE": row["fault_interval_RMSE"],
        "recovery_events": row["recovery_events"], "false_recovery_events": row["false_recovery_events"],
        "substitution_fraction": row["substitution_fraction"],
        "recovery_latency_s": row["sensor_recovery_latency_s"],
        "control_effort_sum_u2": row["control_effort_sum_u2"],
    }

comparison_rows = []
with np.load(project_root / "results/baseline_prefix/final_representative_traces.npz") as baseline_traces:
    for scenario in ["sensor_drift", "combined_fault_load"]:
        prefix = f"D_adaptive__{scenario}__"
        baseline_trace = {name[len(prefix):]: baseline_traces[name] for name in baseline_traces.files if name.startswith(prefix)}
        comparison_rows.append(validation_row("pre_fix", scenario, baseline_trace))
        comparison_rows.append(validation_row("post_fix", scenario, representative_traces[("D_adaptive", scenario)]))
bugfix_scenario_comparison_df = pd.DataFrame(comparison_rows)
bugfix_scenario_comparison_df.to_csv(metrics_dir / "bugfix_scenario_comparison.csv", index=False)

runtime_frames = [horizon_df, warm_df, move_df, iteration_df, profile_df]
runtime_df = pd.concat(runtime_frames, ignore_index=True, sort=False)
final_runtime = final_runs_df[final_runs_df.controller != "A_PI"].copy()
final_runtime["study"] = "final_runtime"
runtime_df = pd.concat([runtime_df, final_runtime], ignore_index=True, sort=False)
runtime_df.to_csv(metrics_dir / "runtime_analysis.csv", index=False)

final_config_dict = fixed_final_config
assert final_config_dict["evaluation_role"] == "untouched_final_holdout"

plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 200,
    "font.size": 10,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "axes.spines.top": False,
    "axes.spines.right": False,
})


def plot_case(scenario, filename):
    fig, axes = plt.subplots(4, 1, figsize=(11, 10), sharex=True, constrained_layout=True)
    axes[0].plot(TIME, reference_values(scenario), "k--", linewidth=1.8, label="Reference")
    for controller in CONTROLLERS:
        trace = representative_traces[(controller, scenario)]
        axes[0].plot(TIME, trace["true"], linewidth=1.25, label=CONTROLLERS[controller])
    axes[0].set_ylabel("Speed (rad/s)")
    axes[0].legend(ncol=2, fontsize=8)

    adaptive = representative_traces[("D_adaptive", scenario)]
    axes[1].plot(TIME, adaptive["measured"], alpha=0.60, label="Measured")
    axes[1].plot(TIME, adaptive["virtual"], "--", linewidth=1.2, label="Virtual LSTM")
    axes[1].plot(TIME, adaptive["feedback"], linewidth=1.1, label="Selected feedback")
    axes[1].set_ylabel("Feedback (rad/s)")
    axes[1].legend(ncol=3, fontsize=8)

    for controller in ["B_plain_MPC", "C_sensor_MPC", "D_adaptive"]:
        trace = representative_traces[(controller, scenario)]
        axes[2].step(TIME, trace["voltage"], where="post", linewidth=1.0, label=CONTROLLERS[controller])
    axes[2].set_ylabel("Voltage (V)")
    axes[2].legend(ncol=3, fontsize=8)

    axes[3].step(TIME, adaptive["substituted"], where="post", label="Virtual feedback used")
    axes[3].step(TIME, adaptive["sensor_state"], where="post", alpha=0.75, label="Sensor suspect")
    axes[3].step(TIME, adaptive["mode"] / 2, where="post", label="Adaptive mode / 2")
    axes[3].set_yticks([0, 0.5, 1.0], ["LOW / trusted", "MEDIUM", "HIGH / suspect"])
    axes[3].set_xlabel("Time (s)")
    axes[3].set_ylabel("Reliability / mode")
    axes[3].legend(ncol=3, fontsize=8)

    config = SCENARIOS[scenario]
    if "event_start" in config:
        for axis in axes:
            axis.axvline(config["event_start"], color="0.25", linestyle=":", linewidth=1)
            if "event_end" in config:
                axis.axvline(config["event_end"], color="0.25", linestyle=":", linewidth=1)
    fig.suptitle(config["title"])
    fig.savefig(plots_dir / filename)
    plt.close(fig)


plot_cases = {
    "step_reference": "final_nominal_step_tracking.png",
    "load_disturbance": "final_load_disturbance.png",
    "bias_15": "final_sensor_bias.png",
    "sensor_dropout": "final_sensor_dropout.png",
    "combined_fault_load": "final_combined_fault_load.png",
}
for scenario, filename in plot_cases.items():
    plot_case(scenario, filename)


def controller_mean(controller, scenarios, metric):
    rows = final_runs_df[
        (final_runs_df.controller == controller)
        & final_runs_df.scenario.isin(scenarios)
    ]
    return float(rows[metric].mean())


def change_text(candidate, baseline, lower_is_better=True):
    percent = 100 * (candidate - baseline) / max(abs(baseline), 1e-9)
    improved = percent < 0 if lower_is_better else percent > 0
    return f"{'improved' if improved else 'worsened'} by {abs(percent):.1f}%"


nominal_scenarios = ["nominal_tracking", "step_reference", "changing_reference"]
sensor_scenarios = ["sensor_noise", "bias_5", "bias_15", "sensor_dropout", "sensor_drift"]
disturbance_scenarios = ["load_disturbance", "parameter_variation", "combined_fault_load"]

plain_nominal_rmse = controller_mean("B_plain_MPC", nominal_scenarios, "RMSE")
pi_nominal_rmse = controller_mean("A_PI", nominal_scenarios, "RMSE")
sensor_fault_rmse = controller_mean("C_sensor_MPC", sensor_scenarios, "fault_interval_RMSE")
plain_fault_rmse = controller_mean("B_plain_MPC", sensor_scenarios, "fault_interval_RMSE")
adaptive_disturbance_rmse = controller_mean("D_adaptive", disturbance_scenarios, "fault_interval_RMSE")
plain_disturbance_rmse = controller_mean("B_plain_MPC", disturbance_scenarios, "fault_interval_RMSE")
adaptive_nominal_rmse = controller_mean("D_adaptive", nominal_scenarios, "RMSE")
sensor_improvement_percent = 100 * (plain_fault_rmse - sensor_fault_rmse) / max(abs(plain_fault_rmse), 1e-9)
adaptive_degradation_percent = 100 * (adaptive_disturbance_rmse - plain_disturbance_rmse) / max(abs(plain_disturbance_rmse), 1e-9)

finite_runtime_ms = final_runtime_samples_df.loc[np.isfinite(final_runtime_samples_df.compute_ms), "compute_ms"].to_numpy()
final_average_ms = float(np.mean(finite_runtime_ms))
final_median_ms = float(np.median(finite_runtime_ms))
all_solve_p95_ms = float(np.percentile(finite_runtime_ms, 95))
all_solve_p99_ms = float(np.percentile(finite_runtime_ms, 99))
final_max_ms = float(np.max(finite_runtime_ms))
over_50_percent = float(100 * np.mean(finite_runtime_ms > 50))
over_100_percent = float(100 * np.mean(finite_runtime_ms > 100))
real_time = final_average_ms <= 50 and all_solve_p95_ms <= 50 and final_max_ms <= 50

runtime_labels = ["Final\nmean", "Final\nmedian", "Final\np95", "Final\nmax"]
runtime_values = [final_average_ms, final_median_ms, all_solve_p95_ms, final_max_ms]
fig, axis = plt.subplots(figsize=(8, 4.8), constrained_layout=True)
bars = axis.bar(runtime_labels, runtime_values, color=["0.55", "#4C78A8", "#F58518", "#E45756"])
axis.axhline(1000 * CONTROL_DT, color="black", linestyle="--", label="50 ms control interval")
axis.bar_label(bars, labels=[f"{value:.1f}" for value in runtime_values], padding=3)
axis.set_ylabel("Solve time (ms)")
axis.set_title("Final MPC solver/control-computation time")
axis.grid(axis="y", alpha=0.25)
axis.spines[["top", "right"]].set_visible(False)
axis.legend()
fig.savefig(plots_dir / "final_runtime_comparison.png", dpi=200)
plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), constrained_layout=True)
axes[0].hist(finite_runtime_ms, bins=50, color="#4C78A8", alpha=0.85)
axes[0].axvline(50, color="black", linestyle="--", label="50 ms deadline")
axes[0].set(xlabel="Solve time (ms)", ylabel="Count", title="Runtime histogram")
ordered_runtime = np.sort(finite_runtime_ms)
axes[1].plot(ordered_runtime, np.arange(1, len(ordered_runtime) + 1) / len(ordered_runtime), color="#4C78A8")
axes[1].axvline(50, color="black", linestyle="--", label="50 ms deadline")
axes[1].set(xlabel="Solve time (ms)", ylabel="Empirical CDF", title="Runtime ECDF")
for axis in axes: axis.legend()
fig.savefig(plots_dir / "final_runtime_distribution.png", dpi=200)
plt.close(fig)

answers = {
    "1_lstm_accuracy": (
        f"Yes for one-step representation: held-out RMSE is {lstm_metrics['test']['rmse']:.3f} rad/s "
        f"with R²={lstm_metrics['test']['r2']:.6f}. Recursive RMSE grows with horizon "
        f"({lstm_metrics['recursive_rmse_by_horizon'][-1]:.3f} rad/s at H=15), so the model is accurate but not uncertainty-free."
    ),
    "2_lstm_mpc_vs_pi": (
        f"Across the three nominal/reference scenarios, plain LSTM-MPC RMSE was {plain_nominal_rmse:.3f} "
        f"versus {pi_nominal_rmse:.3f} rad/s for PI ({change_text(plain_nominal_rmse, pi_nominal_rmse)}). "
        "Scenario rows in final_controller_comparison.csv show where that aggregate result changes."
    ),
    "3_sensor_fault_tolerance": (
        f"Across five sensor-fault scenarios, reliability-aware MPC fault-interval RMSE was {sensor_fault_rmse:.3f} "
        f"versus {plain_fault_rmse:.3f} rad/s for plain MPC ({change_text(sensor_fault_rmse, plain_fault_rmse)})."
    ),
    "4_disturbance_robustness": (
        f"Across load, parameter-variation and combined cases, adaptive MPC fault/recovery RMSE was "
        f"{adaptive_disturbance_rmse:.3f} versus {plain_disturbance_rmse:.3f} rad/s for plain MPC "
        f"({change_text(adaptive_disturbance_rmse, plain_disturbance_rmse)})."
    ),
    "5_nominal_adaptation_tradeoff": (
        f"Adaptive nominal RMSE was {adaptive_nominal_rmse:.3f} versus {plain_nominal_rmse:.3f} rad/s. "
        "When worse, the evidence is consistent with larger move penalties and tighter slew limits delaying corrective voltage; "
        "those restrictions target degraded-condition smoothness and safety, not nominal tracking."
    ),
    "6_selected_horizons": (
        f"The frozen final configuration uses H={FINAL_CONFIG.horizon}, Nc="
        f"{len(FINAL_CONFIG.move_blocks) if FINAL_CONFIG.move_blocks else FINAL_CONFIG.control_horizon}, "
        f"blocks={FINAL_CONFIG.move_blocks}, and max_iterations={FINAL_CONFIG.max_iterations}."
    ),
    "7_runtime_measurement": (
        f"All {len(final_runtime_samples_df)} final MPC compute-control samples are persisted; mean/median/p95/p99/max were "
        f"{final_average_ms:.1f}/{final_median_ms:.1f}/{all_solve_p95_ms:.1f}/{all_solve_p99_ms:.1f}/{final_max_ms:.1f} ms."
    ),
    "8_real_time_feasibility": (
        f"Final solver timing was mean={final_average_ms:.1f} ms, median={final_median_ms:.1f} ms, overall p95={all_solve_p95_ms:.1f} ms, "
        f"max={final_max_ms:.1f} ms against a 50 ms interval. "
        + ("All reported timing summaries meet the interval in this simulation on this machine."
           if real_time else "The current implementation remains simulation-oriented; reliable 20 Hz deployment needs further solver/model optimization or a slower interval.")
    ),
    "9_limitations": (
        "Simultaneous sensor corruption and plant disturbance is the main weak case; adaptive control does not consistently improve it. "
        "The learned plant is open-loop trained; recursive error grows with horizon; the reliability residual can react to plant mismatch; "
        "timings are CPU- and load-dependent; and all evidence is simulation-only."
    ),
    "10_contribution": (
        "The defensible contribution is an evaluation of reliability-aware virtual feedback for constrained LSTM-MPC under faulty sensor measurements. "
        "Plant-mismatch adaptation is secondary and is not claimed to be universally superior; plain MPC remains the benchmark where it performs better."
    ),
}

headline_metrics = {
    "plain_sensor_rmse": plain_fault_rmse,
    "reliable_sensor_rmse": sensor_fault_rmse,
    "sensor_improvement_percent": sensor_improvement_percent,
    "plain_disturbance_rmse": plain_disturbance_rmse,
    "adaptive_disturbance_rmse": adaptive_disturbance_rmse,
    "adaptive_degradation_percent": adaptive_degradation_percent,
}
safety_counts = {
    "optimizer_failures": int(final_runs_df.optimizer_failures.sum()),
    "voltage_violations": int(final_runs_df.voltage_violations.sum()),
    "rate_violations": int(final_runs_df.rate_violations.sum()),
    "nonfinite_events": int(final_runs_df.nonfinite_events.sum()),
}
summary = {
    "selected_configuration": final_config_dict,
    "headline_metrics": headline_metrics,
    "profile": profile_df.to_dict(orient="records"),
    "horizon_control_study": horizon_summary.to_dict(orient="records"),
    "warm_start_study": warm_summary.to_dict(orient="records"),
    "move_blocking_study": move_summary.to_dict(orient="records"),
    "iteration_study": iteration_summary.to_dict(orient="records"),
    "runtime": {
        "measurement": "MPC solver/control-computation time",
        "sample_count": int(len(final_runtime_samples_df)),
        "mean_ms": final_average_ms,
        "median_ms": final_median_ms,
        "p95_ms": all_solve_p95_ms,
        "p99_ms": all_solve_p99_ms,
        "max_ms": final_max_ms,
        "over_50_ms_percent": over_50_percent,
        "over_100_ms_percent": over_100_percent,
        "final_average_ms": final_average_ms,
        "final_median_ms": final_median_ms,
        "final_p95_ms": all_solve_p95_ms,
        "final_max_ms": final_max_ms,
        "control_interval_ms": 1000 * CONTROL_DT,
        "real_time_20_hz": real_time,
    },
    "safety": safety_counts,
    "reliability_validation": {
        "health_residual": "raw_y_measured - y_hat",
        "feedback": "raw_y_measured when trusted; y_hat while substituted",
        "recovery": "finite residual within instant_threshold and CUSUM score within threshold for exit_count consecutive samples",
        "cusum_reset": "only after the complete persistent recovery condition",
        "pi_fallback": fallback_validation,
    },
    "statistical_repetitions": len(FINAL_SEEDS),
    "answers": answers,
}
(metrics_dir / "final_summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

display(Markdown("## Final answers\n\n" + "\n\n".join(
    f"{index}. {answer}" for index, answer in enumerate(answers.values(), 1)
)))

required_files = [
    metrics_dir / "final_controller_runs.csv",
    metrics_dir / "final_controller_comparison.csv",
    metrics_dir / "final_summary.json",
    metrics_dir / "final_runtime_samples.csv",
    metrics_dir / "runtime_analysis.csv",
    metrics_dir / "virtual_feedback_quality.csv",
    metrics_dir / "bugfix_scenario_comparison.csv",
    metrics_dir / "pi_fallback_validation.json",
    configs_dir / "final_mpc_config.json",
    raw_dir / "final_representative_traces.npz",
    *[plots_dir / filename for filename in plot_cases.values()],
    plots_dir / "final_runtime_comparison.png",
    plots_dir / "final_runtime_distribution.png",
]
assert all(path.exists() and path.stat().st_size > 0 for path in required_files)
assert len(final_comparison_df) == len(CONTROLLERS) * len(SCENARIOS)
assert final_comparison_df.repetitions.eq(len(FINAL_SEEDS)).all()
expected_runtime_samples = 3 * len(SCENARIOS) * len(FINAL_SEEDS) * len(range(0, len(TIME), CONTROL_STRIDE))
assert len(final_runtime_samples_df) == expected_runtime_samples
assert not final_runtime_samples_df.duplicated(["controller", "scenario", "seed", "sample_index"]).any()
assert set(final_runs_df.seed) == set(FINAL_SEEDS)
assert all(np.isfinite(trace["true"]).all() and np.isfinite(trace["voltage"]).all() for trace in representative_traces.values())
assert not model.training
print(f"Final checks passed: {len(final_runs_df)} runs, {len(final_comparison_df)} aggregate rows, {len(required_files)} required artifacts.")

## Final answers

1. Yes for one-step representation: held-out RMSE is 0.141 rad/s with R²=0.999906. Recursive RMSE grows with horizon (0.488 rad/s at H=15), so the model is accurate but not uncertainty-free.

2. Across the three nominal/reference scenarios, plain LSTM-MPC RMSE was 8.363 versus 9.348 rad/s for PI (improved by 10.5%). Scenario rows in final_controller_comparison.csv show where that aggregate result changes.

3. Across five sensor-fault scenarios, reliability-aware MPC fault-interval RMSE was 1.374 versus 7.584 rad/s for plain MPC (improved by 81.9%).

4. Across load, parameter-variation and combined cases, adaptive MPC fault/recovery RMSE was 4.710 versus 1.933 rad/s for plain MPC (worsened by 143.7%).

5. Adaptive nominal RMSE was 8.372 versus 8.363 rad/s. When worse, the evidence is consistent with larger move penalties and tighter slew limits delaying corrective voltage; those restrictions target degraded-condition smoothness and safety, not nominal tracking.

6. The frozen final configuration uses H=20, Nc=2, blocks=(5, 15), and max_iterations=8.

7. All 19965 final MPC compute-control samples are persisted; mean/median/p95/p99/max were 68.0/60.6/135.8/166.6/666.9 ms.

8. Final solver timing was mean=68.0 ms, median=60.6 ms, overall p95=135.8 ms, max=666.9 ms against a 50 ms interval. The current implementation remains simulation-oriented; reliable 20 Hz deployment needs further solver/model optimization or a slower interval.

9. Simultaneous sensor corruption and plant disturbance is the main weak case; adaptive control does not consistently improve it. The learned plant is open-loop trained; recursive error grows with horizon; the reliability residual can react to plant mismatch; timings are CPU- and load-dependent; and all evidence is simulation-only.

10. The defensible contribution is an evaluation of reliability-aware virtual feedback for constrained LSTM-MPC under faulty sensor measurements. Plant-mismatch adaptation is secondary and is not claimed to be universally superior; plain MPC remains the benchmark where it performs better.

Final checks passed: 220 runs, 44 aggregate rows, 17 required artifacts.
